# Optional URL Lab: PhiUSIIL Phishing Detection and Thresholding

Companion notebook for *AI for Security and Security for AI*.


## Goal

This optional notebook replaces the unsafe/mismatched PhishTank+Tranco construction with a single labeled URL dataset. The UCI PhiUSIIL Phishing URL Dataset contains legitimate and phishing URL instances with URL/webpage-derived features.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Install/load dataset

The UCI loader may need `ucimlrepo`. If running in Colab, uncomment the installation line.


In [ ]:
# Uncomment in Colab if needed:
# !pip install -q ucimlrepo

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

phiusiil = fetch_ucirepo(id=967)
X = phiusiil.data.features.copy()
y_raw = phiusiil.data.targets.copy()
print(X.shape)
print(y_raw.head())
print(X.dtypes.value_counts())


In [ ]:
# UCI metadata states: label 1 = legitimate URL, label 0 = phishing URL.
# For security convention, convert to phishing-positive:
#   y = 1 means phishing
#   y = 0 means legitimate
target_col = y_raw.columns[0]
y = 1 - y_raw[target_col].astype(int)
X_num = X.select_dtypes(include=[np.number]).copy()
MAX_ROWS = 50000
if len(X_num) > MAX_ROWS:
    sample_idx = X_num.sample(MAX_ROWS, random_state=RANDOM_STATE).index
    X_num = X_num.loc[sample_idx]
    y = y.loc[sample_idx]
X_train, X_temp, y_train, y_temp = train_test_split(X_num, y, test_size=0.40, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE)
print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.value_counts())


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

clf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08, random_state=RANDOM_STATE))
])
clf.fit(X_train, y_train)
val_scores = clf.predict_proba(X_val)[:, 1]
test_scores = clf.predict_proba(X_test)[:, 1]


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, average_precision_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, PrecisionRecallDisplay
)

def binary_metrics(y_true, scores, threshold=0.5):
    y_pred = (scores >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    out = {"threshold": threshold, "accuracy": accuracy_score(y_true, y_pred),
           "precision": p, "recall": r, "f1": f1,
           "pr_auc": average_precision_score(y_true, scores)}
    try:
        out["roc_auc"] = roc_auc_score(y_true, scores)
    except Exception:
        out["roc_auc"] = np.nan
    return out

def choose_threshold(y_true, scores, beta=2.0, min_precision=None):
    best = None
    for th in np.linspace(0.01, 0.99, 99):
        y_pred = (scores >= th).astype(int)
        p, r, _, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
        if min_precision is not None and p < min_precision:
            continue
        beta2 = beta * beta
        fbeta = (1 + beta2) * p * r / (beta2 * p + r + 1e-12)
        if best is None or fbeta > best["fbeta"]:
            best = {"threshold": float(th), "precision": float(p), "recall": float(r), "fbeta": float(fbeta)}
    return best

choice = choose_threshold(y_val.values, val_scores, beta=2.0, min_precision=0.50)
pd.DataFrame([
    binary_metrics(y_test.values, test_scores, threshold=0.5),
    binary_metrics(y_test.values, test_scores, threshold=choice["threshold"])
], index=["default threshold", "validation-tuned threshold"]).round(4)


In [ ]:
threshold = choice["threshold"]
y_pred = (test_scores >= threshold).astype(int)
fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, values_format="d")
ax.set_title(f"PhiUSIIL confusion matrix, threshold={threshold:.2f}")
plt.show()


## Practitioner takeaway

Use a single labeled dataset for URL labs. Mixing phishing URLs from one source with benign domains from another source creates a modality mismatch.
